# The Multi-armed bandit (MAB) problem

- The MAB problem
- The epsilon-greedy method
- Softmax exploration
- The upper confidence bound algorithm
- The Thompson sampling algorithm
- Applications of MAB
- Finding the best advertisement banner using MAB
- Contextual bandits

## The MAB Problem

- A **multi-armed bandit (MAB)** is a slot machine where we pull the arm (lever) and get a payout (reward) based on some probability distribution.
- A single slot machine is known as a **one-armed bandit**.
- When there are multiple slot machines, it's called a **MAB**, or **k-armed bandit**, where $k$ denotes the number of slot machines.
- Whenever we say "arm $n$", we actually mean we're referring to slot machine $n$.
- Each arm has its own probability distribution.
- The probability distribution of each arm is unknown to us — we have to figure out which arm wins us the game most of the time.
- Let $a$ denote an arm, $N$ the number of times that arm has been pulled, and $R_i$ the $i$-th reward received from it. We define the average reward from pulling arm $a$ as:

$$
Q(a) = \frac{1}{N} \sum_{i=1}^{N} R_i
$$

- The **optimal arm** $a^{*}$ is the one that gives the maximum average reward, that is:

$$
a^{*} = \operatorname*{arg\,max}_{a} Q(a)
$$

- We play the game for several rounds, pulling only one arm per round.
- We must, however, minimize the cost of identifying the best arm — this is the classic **exploration-exploitation trade-off**.
- We use the epsilon-greedy method: with probability $1-\epsilon$, we select the arm that has given us the best reward so far; with probability $\epsilon$, we select a random arm instead (to keep exploring).

## Creating a bandit in the Gym

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [ ]:
from envs.bandits.bandit import BanditTwoArmedHighLowFixed

env = BanditTwoArmedHighLowFixed()


#Should be two since we created a 2-armed bandit
print(env.action_space)
print(env.action_space.n)

#Probability distribution
# Win 80% of the time with arm 1
# Win 20% of the time with arm 2
print(env.p_dist)


## Exploration Strategies

- Epsilon-greedy
- Softmax exploration
- Upper confidence bound
- Thomson sampling

### Epsilon-greedy

- Select best arm with probability $1 - \epsilon$
- Select random arm with probability $\epsilon$
- Say we have two arms, arm 1 and arm 2 with the following information

**Arm 1**
- Win the game 80% of the time.

**Arm 2**
- Win the game 20% of the time.

__NB__: Arm 1 is the best arm since it makes us win 80% of the time


In [ ]:
import gymnasium as gym
from envs.bandits.bandit import BanditTwoArmedHighLowFixed
import numpy as np

# Win 80% of the time with arm 0
# Win 20% of the time with arm 1
env = BanditTwoArmedHighLowFixed()
# number of rounds to find best arm
num_rounds = 100
#epsilon
epsilon = 0.5


def epsilon_greedy(env, Q, epsilon=0.1):
    if np.random.uniform(0, 1) < epsilon:
        # explore
        return env.action_space.sample()
    else:
        # exploit
        return np.argmax(Q)


def pull(env, epsilon_greedy, epsilon=0.1, num_rounds=100):
    env.reset()

    n_arms = env.action_space.n
    # number of times an arm is pulled.
    count = np.zeros(n_arms)
    # sum of rewards of each arm
    sum_rewards = np.zeros(n_arms)
    # average reward for each arm
    Q = np.zeros(n_arms)

    for i in range(num_rounds):
        arm = epsilon_greedy(env, Q, epsilon)
        _, reward, _, _, _ = env.step(arm)
        count[arm] += 1
        sum_rewards[arm] += reward
        Q[arm] = sum_rewards[arm] / count[arm]

    print(Q.tolist())
    print(f'The optimal arm is arm {np.argmax(Q)}')

    return Q


pull(env, epsilon_greedy, epsilon, num_rounds)

## Softmax/Boltzmann Exploration

- In epsilon-greedy exploration, all non-best arms are explored **equally** (each gets the same small probability $\epsilon / k$, regardless of how good or bad it actually is).
- To avoid this, we can instead give each arm a priority based on how good it appears to be.
- We assign this priority based on the average reward $Q_t(a)$: the higher an arm's average reward, the higher its probability of being selected.
- Rather than being flatly equal, non-best arms get probabilities that scale with how close their average reward is to the best arm's.

- Formally, this is done using the **softmax (Boltzmann) distribution** over the arms' Q values:

$$
p_t(a) = \frac{e^{Q_t(a)/\tau}}{\sum_{b=1}^{k} e^{Q_t(b)/\tau}}
$$

- $k$ is the total number of arms.
- $\tau$ (tau) is the **temperature** parameter, which controls the amount of exploration:
  - A **high** temperature makes the probabilities across arms closer to uniform (more exploration).
  - A **low** temperature makes the distribution more peaked around the best arm (more exploitation).
  - As $\tau \to 0$, softmax exploration converges to pure greedy selection (always picking $\operatorname*{arg\,max}_{a} Q_t(a)$).
  - We set $\tau$ to a high number in the initial rounds
  - We reduce the value of $\tau$ after a series of rounds.

In [ ]:
import gymnasium as gym
from envs.bandits.bandit import BanditTwoArmedHighLowFixed
import numpy as np

# Win 80% of the time with arm 1
# Win 20% of the time with arm 2
env = BanditTwoArmedHighLowFixed()
# number of rounds to find best arm
num_rounds = 100
# epsilon (unused here, softmax doesn't need it — leftover from epsilon-greedy version)
epsilon = 0.5


def softmax(env, Q, temperature):
    denom = sum([np.exp(i / temperature) for i in Q])
    probs = [np.exp(i / temperature) / denom for i in Q]

    arm = np.random.choice(env.action_space.n, p=probs)
    return arm


def pull(env, softmax, temperature=50, reduction=0.99, num_rounds=100):
    env.reset()

    n_arms = env.action_space.n
    # number of times an arm is pulled.
    count = np.zeros(n_arms)
    # sum of rewards of each arm
    sum_rewards = np.zeros(n_arms)
    # average reward for each arm
    Q = np.zeros(n_arms)

    T = temperature
    for i in range(num_rounds):
        arm = softmax(env, Q, T)
        _, reward, _, _, _ = env.step(arm)
        count[arm] += 1
        sum_rewards[arm] += reward
        Q[arm] = sum_rewards[arm] / count[arm]

        T = T * reduction

    print(Q.tolist())
    print(f'The optimal arm is arm {np.argmax(Q)}')

    return Q


pull(env, softmax, temperature=50, reduction=0.99, num_rounds=num_rounds)

## Upper Confidence Bound (UCB)

- Based on the **optimism in the face of uncertainty** principle.
- The confidence interval denotes the interval within which the true value lies — that is, the interval within which the true mean reward of the arm falls.
- For an interval $[0.2, 0.9]$, $0.2$ is the **lower confidence bound** and $0.9$ is the **upper confidence bound**.
- When the confidence interval is large, we are uncertain about the mean value.
- When the confidence interval is small, we are more certain about the mean value.
- A smaller interval implies that the arm has been explored a lot more (we have more data on it, so our estimate is tighter).
- In UCB, we pull the arm with the highest upper confidence bound — this naturally balances exploitation (favoring arms with high estimated reward) with exploration (favoring arms we're still uncertain about).

- The upper confidence bound for arm $a$ at round $t$ is given by:

$$
UCB_t(a) = Q_t(a) + c \sqrt{\frac{\ln t}{N_t(a)}}
$$

- $Q_t(a)$ is the average reward obtained from arm $a$ so far (the exploitation term).
- $N_t(a)$ is the number of times arm $a$ has been pulled so far.
- $t$ is the current round (time step) number.
- $c$ is a constant that controls the degree of exploration.
- The term $c\sqrt{\ln t / N_t(a)}$ is the exploration bonus: it shrinks as $N_t(a)$ grows (an arm pulled often becomes well-understood, so its bonus shrinks), and it grows slowly with $t$ (so arms neglected for a long time regain a bit of their exploration bonus over time).

- We then select the arm with the highest upper confidence bound:

$$
a^{*} = \operatorname*{arg\,max}_{a} \; UCB_t(a)
$$

In [ ]:
import gymnasium as gym
from envs.bandits.bandit import BanditTwoArmedHighLowFixed
import numpy as np

# Win 80% of the time with arm 1
# Win 20% of the time with arm 2
env = BanditTwoArmedHighLowFixed()
# number of rounds to find best arm
num_rounds = 100


def UCB(env, Q, count, i):
    n_arms = env.action_space.n
    ucb = np.zeros(n_arms)

    for arm in range(n_arms):
        if count[arm] == 0:
            # force every arm to be tried at least once first
            ucb[arm] = float('inf')
        else:
            ucb[arm] = Q[arm] + np.sqrt((2 * np.log(sum(count))) / count[arm])

    return np.argmax(ucb)


def pull(env, UCB, num_rounds=100):
    env.reset()

    n_arms = env.action_space.n
    # number of times an arm is pulled.
    count = np.zeros(n_arms)
    # sum of rewards of each arm
    sum_rewards = np.zeros(n_arms)
    # average reward for each arm
    Q = np.zeros(n_arms)

    for i in range(num_rounds):
        arm = UCB(env, Q, count, i)
        _, reward, _, _, _ = env.step(arm)
        count[arm] += 1
        sum_rewards[arm] += reward
        Q[arm] = sum_rewards[arm] / count[arm]

    print(Q.tolist())
    print(f'The optimal arm is arm {np.argmax(Q)}')

    return Q


pull(env, UCB, num_rounds=num_rounds)

## Thompson Sampling

- A **Bayesian** approach to the exploration-exploitation trade-off in the multi-armed bandit problem.
- Unlike epsilon-greedy, softmax, and UCB — which use a single point estimate of each arm's average reward — Thompson sampling maintains a full **probability distribution** over how likely each arm is to be the best one.
- The core idea: instead of committing to the arm with the highest *estimated* reward, we sample a plausible reward value from each arm's distribution and pull whichever arm's sample happens to be highest. This naturally balances exploration and exploitation — arms we're uncertain about have wide distributions and can occasionally produce a high sample, while arms we're confident are good keep winning consistently.

### The Beta Distribution

- Thompson sampling is most commonly used for **Bernoulli bandits** — arms that pay out a reward of 1 (success) or 0 (failure) with some unknown probability.
- For this setting, we model our belief about each arm's true success probability using a **Beta distribution**:

$$
\text{Beta}(\alpha, \beta)
$$

- $\alpha$ can be thought of as (roughly) 1 + the number of successes (rewards of 1) observed from the arm so far.
- $\beta$ can be thought of as (roughly) 1 + the number of failures (rewards of 0) observed from the arm so far.
- The Beta distribution is a good fit here because it is defined over the interval $[0, 1]$ — exactly the range of a probability — and it is the **conjugate prior** of the Bernoulli distribution, meaning the posterior after observing new data is also a Beta distribution, making the updates simple.
- Before any data is observed, each arm starts with $\text{Beta}(1, 1)$, which is just the uniform distribution over $[0, 1]$ — i.e., we assume nothing about which arm is better.

### Updating the Distribution

- After pulling arm $a$ and observing reward $r \in \{0, 1\}$, we update that arm's distribution:

$$
\alpha_a \leftarrow \alpha_a + r
$$

$$
\beta_a \leftarrow \beta_a + (1 - r)
$$

- In words: a success ($r=1$) increases $\alpha_a$ by 1; a failure ($r=0$) increases $\beta_a$ by 1.
- As more data is collected for an arm, its Beta distribution becomes narrower and more peaked around the arm's true success probability — reflecting growing confidence.

### The Thompson Sampling Algorithm

1. Initialize $\alpha_a = 1$ and $\beta_a = 1$ for every arm $a$.
2. For each round $t$:
   1. For every arm $a$, draw a sample $\hat{\theta}_a \sim \text{Beta}(\alpha_a, \beta_a)$.
   2. Pull the arm with the highest sampled value:

$$
a^{*} = \operatorname*{arg\,max}_{a} \; \hat{\theta}_a
$$

   3. Observe the reward $r$ from pulling arm $a^{*}$.
   4. Update that arm's distribution: $\alpha_{a^{*}} \leftarrow \alpha_{a^{*}} + r$ and $\beta_{a^{*}} \leftarrow \beta_{a^{*}} + (1-r)$.
3. Repeat for many rounds — over time, the distributions of good arms narrow around high values and get sampled (and pulled) more often, while poor arms are naturally explored less and less.

### Why It Works

- Early on, all arms have wide, overlapping distributions (little data), so sampled values vary a lot and every arm gets a fair chance of being pulled — this drives **exploration**.
- As data accumulates, the distribution of a consistently good arm narrows around a high value, so it gets sampled highest most of the time — this drives **exploitation**.
- Unlike epsilon-greedy, there is no separate exploration probability $\epsilon$ or temperature $\tau$ to tune — the exploration/exploitation balance emerges naturally from how much data each arm has collected.

In [ ]:
import gymnasium as gym
from envs.bandits.bandit import BanditTwoArmedHighLowFixed
import numpy as np

# Win 80% of the time with arm 1
# Win 20% of the time with arm 2
env = BanditTwoArmedHighLowFixed()
# number of rounds to find best arm
num_rounds = 100


def thompson_sampling(env, alpha, beta):
    n_arms = env.action_space.n
    samples = [np.random.beta(alpha[arm], beta[arm]) for arm in range(n_arms)]

    return np.argmax(samples)


def pull(env, thompson_sampling, num_rounds=100):
    env.reset()

    n_arms = env.action_space.n
    # number of times an arm is pulled.
    count = np.zeros(n_arms)
    # sum of rewards of each arm
    sum_rewards = np.zeros(n_arms)
    # average reward for each arm
    Q = np.zeros(n_arms)
    # beta distribution parameters for each arm, start uninformative (uniform prior)
    alpha = np.ones(n_arms)
    beta = np.ones(n_arms)

    for i in range(num_rounds):
        arm = thompson_sampling(env, alpha, beta)
        _, reward, _, _, _ = env.step(arm)
        count[arm] += 1
        sum_rewards[arm] += reward
        Q[arm] = sum_rewards[arm] / count[arm]

        # update beta distribution based on observed reward
        if reward == 1:
            alpha[arm] += 1
        else:
            beta[arm] += 1

    print(Q.tolist())
    print(f'The optimal arm is arm {np.argmax(Q)}')

    return Q


pull(env, thompson_sampling, num_rounds=num_rounds)

## Consolidated Algorithms

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [ ]:
# Epsilon-Greedy

import gymnasium as gym
from envs.bandits.bandit import BanditTwoArmedHighLowFixed
from envs.bandits.algorithms import BanditSolver

env = BanditTwoArmedHighLowFixed()
solver = BanditSolver(env, algorithm='epsilon-greedy', epsilon=0.5)
solver.pull(num_rounds=100)
print(solver.Q)
print(solver.get_optimal_arm())

In [ ]:
# softmax
from envs.bandits.bandit import BanditTwoArmedHighLowFixed
from envs.bandits.algorithms import BanditSolver

env = BanditTwoArmedHighLowFixed()
solver = BanditSolver(env, algorithm='softmax', temperature=50, reduction=0.99, constant=False)
solver.pull(num_rounds=100)
print(solver.Q)
print(solver.get_optimal_arm())

In [ ]:
# ucb
from envs.bandits.bandit import BanditTwoArmedHighLowFixed
from envs.bandits.algorithms import BanditSolver

env = BanditTwoArmedHighLowFixed()
solver = BanditSolver(env, algorithm='ucb')
solver.pull(num_rounds=100)
print(solver.Q)
print(solver.get_optimal_arm())

In [ ]:
# thompson
from envs.bandits.bandit import BanditTwoArmedHighLowFixed
from envs.bandits.algorithms import BanditSolver

env = BanditTwoArmedHighLowFixed()
solver = BanditSolver(env, algorithm='thompson', alpha=1, beta=1)
solver.pull(num_rounds=100)
print(solver.Q)
print(solver.get_optimal_arm())